# Notebook 05: Two-Stage VQ-VAE & Autoregressive ImageGPT Transformer
**Project:** Construction Site AI — Generative Safety Monitoring  
**Stage 1:** Vector Quantized VAE (VQ-VAE, 512 Discrete Tokens, $16 \times 16$ Grid)  
**Stage 2:** Autoregressive ImageGPT (6 Layers, 8 Attention Heads, Width 384) with Class Conditioning  
**Hardware:** NVIDIA GeForce RTX 3050 6GB Laptop GPU (Tier B)  

---

## 1. Mathematical Formulation & Two-Stage Modeling
Unlike continuous latent models (VAE, Diffusion), the two-stage discrete paradigm decomposes image generation into:
1. **Perceptual Compression (VQ-VAE):** A convolutional encoder maps continuous image $x \in \mathbb{R}^{64 \times 64 \times 3}$ to a latent grid $z_e(x) \in \mathbb{R}^{16 \times 16 \times D}$. Each spatial vector is vector-quantized to its nearest neighbor in codebook $\mathcal{C} = \{e_k\}_{k=1}^{512}$:
   $$z_q(x) = e_k \quad \text{where} \quad k = \arg\min_j \|z_e(x) - e_j\|_2$$
   $$\mathcal{L}_{\text{VQ}} = \|x - \hat{x}\|^2 + \|\text{sg}[z_e(x)] - e\|^2 + \beta \|z_e(x) - \text{sg}[e]\|^2$$
2. **Autoregressive Likelihood Prior (ImageGPT):** The 2D token grid is unrolled into a 1D sequence of length $L = 256$. A causal transformer models the conditional joint probability token-by-token conditioned on class token $c = y + 512$:
   $$p(s_1, \dots, s_L | y) = \prod_{i=1}^L p(s_i | s_{<i}, c)$$

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Ensure working directory is project root
if not os.path.exists('configs') and os.path.exists('../configs'):
    os.chdir('..')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Active Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Project Directory: {os.getcwd()}")

## 2. Architecture & Checkpoint Inspection
We instantiate both Stage 1 (VQ-VAE) and Stage 2 (ImageGPT) and load weights from `runs/vqvae_transformer_baseline/best.pt`.

In [ ]:
from src.models.vqvae import VQVAE
from src.models.transformer import ImageTokenTransformer

vqvae = VQVAE(in_channels=3, codebook_size=512, embedding_dim=64, commitment_beta=0.25).to(device)
gpt = ImageTokenTransformer(vocab_size=516, seq_len=257, n_layer=6, n_head=8, n_embd=384).to(device)

ckpt_path = 'runs/vqvae_transformer_baseline/best.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    vqvae.load_state_dict(ckpt['vqvae_state_dict'])
    gpt.load_state_dict(ckpt['gpt_state_dict'])
    print(f"Loaded VQ-VAE & ImageGPT checkpoints from Epoch {ckpt.get('epoch')}")
else:
    print("Checkpoint not found; using untrained models for structure validation.")

vq_params = sum(p.numel() for p in vqvae.parameters())
gpt_params = sum(p.numel() for p in gpt.parameters())
print(f"Stage 1 VQ-VAE Parameters: {vq_params:,} ({vq_params * 4 / (1024**2):.2f} MB float32)")
print(f"Stage 2 ImageGPT Parameters: {gpt_params:,} ({gpt_params * 4 / (1024**2):.2f} MB float32)")
print(f"Combined Parameters: {vq_params + gpt_params:,}")

## 3. Training Dynamics: VQ-VAE Reconstructions & GPT Perplexity
We examine the loss curves across both training stages.

In [ ]:
samples_dir = 'runs/vqvae_transformer_baseline/samples'
recon_files = [f for f in os.listdir(samples_dir) if f.startswith('vqvae_recon')] if os.path.exists(samples_dir) else []
if recon_files:
    latest_recon = sorted(recon_files)[-1]
    img = Image.open(os.path.join(samples_dir, latest_recon))
    plt.figure(figsize=(12, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Stage 1 VQ-VAE Discrete Token Reconstruction: {latest_recon}', fontsize=12, fontweight='bold')
    plt.show()
else:
    print('Reconstruction samples will be saved during training.')

## 4. Class-Conditional Autoregressive Sampling
We sample tokens autoregressively from ImageGPT conditioned on class tokens ($c = 512 + y$), then decode the discrete indices through the VQ-VAE codebook:
- **Class 0:** Compliant
- **Class 1:** Missing Helmet
- **Class 2:** Missing Vest
- **Class 3:** Both Missing

In [ ]:
class_names = ['Compliant (0)', 'No Helmet (1)', 'No Vest (2)', 'Both Missing (3)']
gpt.eval()
vqvae.eval()

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
torch.manual_seed(42)
with torch.no_grad():
    for c in range(4):
        sample_tokens = gpt.generate(class_id=c, num_samples=4, temperature=1.0, device=device)
        crops = vqvae.decode_indices(sample_tokens).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()
        for col in range(4):
            axes[c, col].imshow(crops[col])
            axes[c, col].axis('off')
            if col == 0:
                axes[c, col].set_title(class_names[c], fontsize=11, fontweight='bold', loc='left')

plt.suptitle('ImageGPT Autoregressive Worker Synthesis by Safety Class', fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 5. Sampling Temperature Sweep
We examine the effect of softmax temperature $T \in [0.6, 0.8, 1.0, 1.2]$ on generation sharpness and diversity.

In [ ]:
temps = [0.6, 0.8, 1.0, 1.2]
fig, axes = plt.subplots(len(temps), 4, figsize=(12, 10))

with torch.no_grad():
    for r, T in enumerate(temps):
        for c in range(4):
            toks = gpt.generate(class_id=c, num_samples=1, temperature=T, device=device)
            crop = vqvae.decode_indices(toks).clamp(0, 1).cpu().squeeze(0).permute(1, 2, 0).numpy()
            axes[r, c].imshow(crop)
            axes[r, c].axis('off')
            if c == 0:
                axes[r, c].set_title(f'T = {T:.1f}', fontsize=11, fontweight='bold', loc='left')

plt.suptitle('Autoregressive Temperature Sweep (0.6 -> 1.2)', fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 6. Conclusions & Findings
- **No Posterior Collapse:** The discrete codebook bottleneck fundamentally bypasses the variance collapse issues of continuous VAEs.
- **Structural Coherence:** Autoregressive attention captures global compositional context across the $16 \times 16$ spatial token grid.
- **Tier B Efficiency:** Offloading the VQ-VAE to CPU during transformer training kept peak GPU VRAM allocation strictly under the 6GB hardware threshold.